In [1]:
import pandas as pd
from config import get_engine
engine = get_engine()

pd.read_sql("SELECT TOP 5 * FROM sys.tables", engine)

,name,object_id,principal_id,schema_id,parent_object_id,type,type_desc,create_date,modify_date,is_ms_shipped,...,history_retention_period_unit_desc,is_node,is_edge,data_retention_period,data_retention_period_unit,data_retention_period_unit_desc,ledger_type,ledger_type_desc,ledger_view_id,is_dropped_ledger_table
0,sales,901578250,None,1,0,U,USER_TABLE,2025-12-12 09:21:59.107,2025-12-12 09:21:59.107,False,...,None,False,False,-1,-1,INFINITE,0,NON_LEDGER_TABLE,None,False
1,menu,917578307,None,1,0,U,USER_TABLE,2025-12-12 09:22:13.127,2025-12-12 09:22:13.127,False,...,None,False,False,-1,-1,INFINITE,0,NON_LEDGER_TABLE,None,False
2,members,933578364,None,1,0,U,USER_TABLE,2025-12-12 09:22:20.880,2025-12-12 09:22:20.880,False,...,None,False,False,-1,-1,INFINITE,0,NON_LEDGER_TABLE,None,False


In [2]:
import pandas as pd
import numpy as np
from config import get_engine
engine=get_engine()
sales=pd.read_sql("select * from sales", engine)
menu= pd.read_sql("select * from menu", engine)
members=pd.read_sql("select * from members", engine)

In [18]:
sales["customer_id"]=sales["customer_id"].str.upper()

## 1. Total amount spent by each customer


In [54]:
sales= (sales.merge(menu, on="product_id").groupby('customer_id')['price'].sum())
print(sales)

customer_id
A    76
B    74
C    36
Name: price, dtype: int64


## -- 2. Number of distinct visit days per customer


In [59]:
sales = pd.read_sql("SELECT * FROM sales", engine)

sale=(sales.groupby('customer_id', as_index=False)['order_date'].nunique())
sale


,customer_id,order_date
0,A,4
1,B,6
2,C,2


## -- 3. First item purchased by each customer


In [84]:
df= (sales.merge(menu, on= 'product_id', how='inner').sort_values(['customer_id','order_date']).groupby('customer_id', as_index=False).first())
result=df[["customer_id","product_name","price"]]
print(result)

  customer_id product_name  price
0           A      biryani     10
1           B       paneer     15
2           C        dosai     12


## --4. Most purchased item and count

In [11]:
df=sales.merge(menu, on ='product_id', how= 'inner').groupby('product_name').size().reset_index(name='purchase_count')
result=df.sort_values('purchase_count', ascending=False).head(1)
result

,product_name,purchase_count
1,dosai,8


## -- 5. Most popular item per customer

In [20]:
df= sales.merge(menu, on='product_id', how='inner').groupby(['customer_id', 'product_name']).size().reset_index(name='purchase_count')
##result = df.loc[df.groupby('customer_id')['purchase_count'].idxmax()]
result = (
    df
    .sort_values(['customer_id', 'purchase_count'], ascending=[True, False])
    .drop_duplicates('customer_id')
)
result

,customer_id,product_name,purchase_count
1,A,dosai,3
3,B,biryani,2
6,C,dosai,3


## -- 6. First item after becoming a member


In [30]:
df=(sales.merge(members, on='customer_id', how='inner')
    .merge(menu, on='product_id', how='inner'))

df1 = df[df['order_date'] >= df['join_date']]

result = (
    df1
    .sort_values(['customer_id', 'order_date'])
    .groupby('customer_id', as_index=False)
    .first()[['customer_id', 'product_name']]
)

print(result)

  customer_id product_name
0           A       paneer
1           B      biryani


## -- 7.Last item before becoming a member

In [35]:
df=(sales.merge(members, on='customer_id', how='inner')
    .merge(menu, on='product_id', how='inner')
   )
df1= df[df['order_date']<df['join_date']]

result= (
    df1.sort_values(['customer_id', 'order_date'])
    .groupby('customer_id', as_index=False)
    .last()[['customer_id', 'product_name', 'order_date']]
)
print(result)

  customer_id product_name  order_date
0           A       paneer  2025-01-01
1           B      biryani  2025-01-04


In [ ]:
-- 8. Items and amount before becoming a member

In [ ]:
-- 9. Loyalty points: 2x for biryani, 1x for others

In [ ]:
-- 10. Points during first 7 days after joining

In [ ]:
-- 11. Total spent on biryani

In [ ]:
-- 12. Customer with most dosai orders

In [ ]:
-- 13. Average spend per visit

In [ ]:
-- 14. Day with most orders in Jan 2025

In [ ]:
-- 15. Customer who spent the least

In [ ]:
-- 16. Date with most money spent

In [ ]:
-- 17. Customers with multiple orders on same day

In [ ]:
-- 18. Visits after membership

In [ ]:
-- 19. Items never ordered